In [6]:
from typing import List
import re
import math
import pandas as pd
import os
import json
from transformers import AutoTokenizer, AutoModel

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
input_path = "SGK_Tin12_CD_clean.md"
with open(input_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

In [10]:
import re

output_path = "SGK_Tin12_CD_clean.md"

# Đọc file gốc
with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

clean_text = text

# Xóa các dòng dạng ## Page 42, ### Page 1, # Page 3
clean_text = re.sub(
    r'^\s*#{1,3}\s*Page\s*\d+\s*$', 
    '', 
    clean_text, 
    flags=re.IGNORECASE | re.MULTILINE
)

# Xóa các dòng dạng **Page 35**, *Page 7*, Page 9
clean_text = re.sub(
    r'^\**\s*Page\s*\d+\s*\**$', 
    '', 
    clean_text, 
    flags=re.MULTILINE
)

# Xóa các dòng dạng # --- Trang 7 --- → chuyển thành ---
clean_text = re.sub(
    r'^\s*#\s*-{3}\s*Trang\s*\d+\s*-{3}\s*$',
    '---',
    clean_text,
    flags=re.MULTILINE
)

# Xóa **Trang 12**
clean_text = re.sub(r'^\*\*Trang \d+\*\*\s*$', '', clean_text, flags=re.MULTILINE)

# Xóa # Trang 12
clean_text = re.sub(r'^# Trang \d+\s*$', '', clean_text, flags=re.MULTILINE)

# Xóa ## Trang 7 (pattern mới bổ sung)
clean_text = re.sub(r'^\s*#{2,3}\s*Trang\s*\d+\s*$', '', clean_text, flags=re.MULTILINE)


# Xóa dòng trống thừa sau khi xóa
clean_text = re.sub(r'\n\s*\n+', '\n', clean_text)

# Ghi ra file mới
with open(output_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("✅ Đã loại bỏ tất cả pattern dạng Page và Trang.")


✅ Đã loại bỏ tất cả pattern dạng Page và Trang.


In [12]:
import re
from typing import List, Dict
import json
from collections import Counter

class MarkdownHierarchicalChunker:
    """
    Class xử lý Markdown thành hierarchical chunks.
    Hỗ trợ:
    - Phân tích heading để tạo hierarchy (Parent -> Child)
    - Tạo chunk theo section, merge section ngắn với child nếu cần
    - Semantic chunking: tách nội dung thành các đoạn nhỏ dựa trên số ký tự
    """

    def __init__(self, min_chunk_size: int = 100, merge_short_sections: bool = True):
        """
        min_chunk_size: số ký tự tối thiểu để coi section là "dài"
        merge_short_sections: nếu True, merge section ngắn với các child
        """
        self.min_chunk_size = min_chunk_size
        self.merge_short_sections = merge_short_sections

    def parse_markdown(self, markdown_text: str) -> List[Dict]:
        """
        Phân tích markdown theo từng dòng, phát hiện headings (# đến ######)
        Trả về danh sách section với level, title, content, start_line, end_line
        """
        lines = markdown_text.split('\n')
        sections = []
        current_section = None

        for i, line in enumerate(lines):
            # Match heading markdown: # Heading, ## Subheading...
            heading_match = re.match(r'^(#{1,6})\s+(.+)$', line)
            if heading_match:
                if current_section:
                    current_section['end_line'] = i - 1
                    sections.append(current_section)
                level = len(heading_match.group(1))  # số # = level
                title = heading_match.group(2).strip()
                current_section = {
                    'level': level,
                    'title': title,
                    'content': '',
                    'start_line': i,
                    'end_line': i
                }
            else:
                if current_section:
                    current_section['content'] += line + '\n'

        if current_section:
            current_section['end_line'] = len(lines) - 1
            sections.append(current_section)

        return sections

    def build_hierarchy(self, sections: List[Dict]) -> List[Dict]:
        """
        Xây dựng hierarchical structure từ danh sách sections phẳng.
        Parent section chứa danh sách child nếu level nhỏ hơn.
        """
        hierarchy = []
        stack = []

        for section in sections:
            # Nếu stack top >= level mới → pop ra, tìm parent thích hợp
            while stack and stack[-1]['level'] >= section['level']:
                stack.pop()
            if stack:
                if 'children' not in stack[-1]:
                    stack[-1]['children'] = []
                stack[-1]['children'].append(section)
            else:
                hierarchy.append(section)
            stack.append(section)

        return hierarchy

    def semantic_chunk(text, max_chars=1000, overlap_sentences=2):
        """
        Tách text thành các đoạn nhỏ (semantic chunks) dựa trên số ký tự
        overlap_sentences: số câu overlap giữa các chunk liên tiếp
        """
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks = []
        current_chunk = []
        current_len = 0

        for i, sentence in enumerate(sentences):
            current_chunk.append(sentence)
            current_len += len(sentence) + 1  # +1 cho khoảng trắng

            if current_len >= max_chars:
                chunks.append(' '.join(current_chunk))
                # giữ lại vài câu overlap cho chunk tiếp theo
                current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences < len(current_chunk) else current_chunk
                current_len = sum(len(s) + 1 for s in current_chunk)

        if current_chunk:
            chunks.append(' '.join(current_chunk))

        return chunks

    def merge_section_with_children(self, section: Dict) -> str:
        """
        Merge section ngắn với tất cả child content.
        Giữ tiêu đề child header để dễ hiểu context
        """
        merged_content = section['content'].strip()
        if 'children' in section and section['children']:
            for child in section['children']:
                child_header = f"\n\n{'#' * child['level']} {child['title']}\n"
                merged_content += child_header + child['content'].strip()
        return merged_content

    def create_chunks(self, hierarchy: List[Dict], parent_context: str = "") -> List[Dict]:
        """
        Tạo chunk từ hierarchy
        - Sections ngắn + có child → merge với child
        - Sections ngắn + không có child → chunk riêng
        - Sections dài → chunk riêng
        - Ghi nhận context hierarchical để embedding/search
        """
        chunks = []

        for section in hierarchy:
            # Tạo hierarchical context
            current_context = parent_context
            if current_context:
                current_context += f" > {'#' * section['level']} {section['title']}"
            else:
                current_context = f"{'#' * section['level']} {section['title']}"

            content = section['content'].strip()
            is_short_section = len(content) < self.min_chunk_size
            has_children = 'children' in section and section['children']

            # Merge section ngắn với child nếu cần
            if is_short_section and has_children and self.merge_short_sections:
                merged_content = self.merge_section_with_children(section)
                chunks.append({
                    'context': current_context,
                    'content': merged_content,
                    'metadata': {
                        'level': section['level'],
                        'title': section['title'],
                        'type': 'merged_with_children'
                    }
                })
                continue
            # Section ngắn không có child
            elif is_short_section and not has_children:
                if content:
                    chunks.append({
                        'context': current_context,
                        'content': content,
                        'metadata': {
                            'level': section['level'],
                            'title': section['title'],
                            'type': 'short_section'
                        }
                    })
            else:
                # Section dài → chunk bình thường
                if content:
                    chunks.append({
                        'context': current_context,
                        'content': content,
                        'metadata': {
                            'level': section['level'],
                            'title': section['title'],
                            'type': 'normal'
                        }
                    })
                # Tạo chunk cho child sections (đệ quy)
                if has_children:
                    child_chunks = self.create_chunks(section['children'], current_context)
                    chunks.extend(child_chunks)

        return chunks

    def is_tutorial_heading(line: str) -> bool:
        """
        Kiểm tra xem dòng có phải heading tutorial kiểu #### 1. ... không
        """
        pattern = r'^\s*####\s*\d+\.\s*'
        return bool(re.match(pattern, line))

    def chunk_markdown(self, markdown_text: str) -> List[Dict]:
        """
        Hàm chính để chuyển markdown text → list of chunks
        - Phân tích markdown
        - Build hierarchy
        - Tạo chunks theo hierarchy
        """
        sections = self.parse_markdown(markdown_text)
        hierarchy = self.build_hierarchy(sections)
        chunks = self.create_chunks(hierarchy)
        return chunks


In [ ]:
input_path = "SGK_Tin12_CD_clean.md"
with open(input_path, "r", encoding="utf-8") as f:
    markdown_text = f.read()

chunker = MarkdownHierarchicalChunker(min_chunk_size=100, merge_short_sections=True)
chunks = chunker.chunk_markdown(markdown_text)

print(f"✅ Tổng số chunks: {len(chunks)}\n")

types = Counter([c['metadata']['type'] for c in chunks])
print("📊 Phân loại chunks:")
for chunk_type, count in types.items():
    print(f"  - {chunk_type}: {count}")
print()

for i, chunk in enumerate(chunks[:5], 1):
    print(f"\n[CHUNK {i}] Type: {chunk['metadata']['type']}")
    print(f"Context: {chunk['context']}")
    print(f"Level: {chunk['metadata']['level']} | Title: {chunk['metadata']['title']}")
    print(f"Content length: {len(chunk['content'])} chars")
    preview = chunk['content'][:300] + "..." if len(chunk['content']) > 300 else chunk['content']
    print(f"\nContent preview:\n{preview}")
    print("-"*60)

output_path = "rag_chunks.json"
with open(output_path, "a", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã lưu {len(chunks)} chunks vào: {output_path}")


In [2]:
import pandas as pd

df = pd.read_csv('../data/du_lieu_mapped.csv')
df_unique = df.drop_duplicates()
df_unique.to_csv('../data/du_lieu_mapped1.csv', index=False)